# Introduction to Brightway — Part 2

In this section, we will discuss the fundamental concepts of Brightway. It is important to note that all this information is available online in the documentation page:

https://docs.brightway.dev/en/latest/index.html

This notebook will run in a new session, so we need to install the dependencies once again.

In [ ]:
!pip install bw2calc>=2.1 -q # Brightway package
!pip install bw2data>=4.5 -q # Brightway package
!pip install bw2io>=0.9.11 -q # Brightway package
!pip install polars -q
!pip install pypardiso -q
!pip install seaborn>=0.13.2 -q

<div class="alert alert-block alert-warning">
⚠️ You must restart the session!
</div>

We need to download a backup file that contains ecoinvent. To do this, we need to authenticate the **Gmail** user that was created previously.

In [ ]:
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
drive.CreateFile({'id': '1y7qVWU-G9XDKfOH-QbF2ikREnRjLRXV7'}).GetContentFile('backup.tar.gz')

Check the file.

In [ ]:
!du -hs backup.tar.gz

In [ ]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from rich import print

### Import the project backup
This approach does not require much explanation: the project is loaded again.

In [ ]:
bi.restore_project_directory(
    'backup.tar.gz',  # nombre del archivo, creado celdas arriba
    project_name='proyecto_ei', # Se puede elegir un nombre nuevo para el proyecto
    overwrite_existing = False
    )

## Importing commercial databases
We have learned how to build an LCA model from scratch and manually. Although this is quite useful, in practice we usually combine our own data with data from commercial databases. In this section, we focus on the ecoinvent database (v3.9), one of the most widely used databases in the field.

There are currently two ways to import ecoinvent data into our project:
- Reading the raw ecospold2 files directly from disk and converting them into a Brightway database.
- Using the `import_ecoinvent_release` tool, which downloads the database from a remote server.

### Importing raw ecoinvent data from disk

For this option, ecoinvent must already have been downloaded. Ecoinvent is distributed as a compressed 7z archive and contains all activities in ecospold2 format, which is similar to XML. `bw2io` includes functions designed to interpret the information, verify that the `exchanges` are correct, and check that the biosphere nodes exist in the `biosphere3` database.

In [ ]:
# Para importar, hay que seguir los siguentes pasos:
# 1. Leer los archivos XML e dejar que brigthway los interprete.
# db = bi.SingleOutputEcospold2Importer(dirpath='<datasets-folder>',db_name='ecoinvent39')

In [ ]:
# 2. Aplicar una serie de estrategias para asegurarse que no existe informacion corrupta y que la importacion es posible
# db.apply_strategies()

In [ ]:
# 3. Ecoinvent esta listo en la memoria pero aun no ha sido grabado en el disco.
# Hay que grabarlo en el disco.
# db.write_database()

To verify that it has been imported correctly, we can repeat the exercise performed with the `biosphere3` database in the previous section.

In [ ]:
# bd.databases # Lista de las bases de datos

In [ ]:
# ei = bd.Database('ecoinvent39')
# len(ei) # Muestra la cantidad de elementos

### Importing ecoinvent from a remote server
For this option, we use the `bw2io.import_ecoinvent_release` function, which handles: 1) installing a biosphere, 2) installing the most recent impact assessment methods, and 3) installing the ecoinvent database.
As you can imagine, this requires user authentication with an ecoinvent access account.

In [ ]:
# bw2io.import_ecoinvent_release(
#     version="3.9"
#     system_model="cutoff", # Otras opciones son: "consequential", "apos" y "EN15804"
#     username="xxxx", # Tu usuario
#     password="xxxx", # Tu clave
#     biosphere_name="biosphere" # Optional, puedes guardar la base de datos de la biosfera con otro nombre.
# )

### Exploring ecoinvent

In [ ]:
import bw2data as bd
bd.projects.set_current("proyecto_ei")
ei = bd.Database('ecoinvent-3.9.1-cutoff')
seleccionado = ei.random() # Explora las actividades
# First we inspect the keys of the selected activity
print(list(seleccionado.keys()))

In [ ]:
# We can view the content of the whole dataset.
print(seleccionado.as_dict())

As you can see, the content of an ecoinvent activity is quite rich. There are fields beyond `name`, `code`, `location`, and `unit` that are new to us. This shows that Brightway is flexible enough when defining an activity.

What we saw in the previous cell describes an activity, but it does not yet describe its connections (`exchanges`). To access them, use the `exchanges`, `technosphere`, or `biosphere` functions, depending on what you want to inspect.

In [ ]:
# `exchanges` retorna un objeto the brightway que no es nativo de python
type(seleccionado.exchanges())

In [ ]:
# If we want to read it like a list, we need to convert it into a list.
for exchange in seleccionado.exchanges():
    print(exchange)
# print(list(seleccionado.exchanges()))

In [ ]:
# If we want only the technosphere, we use the corresponding function
for exchange in seleccionado.technosphere():
    print(exchange)
# print(list(seleccionado.technosphere()))

The output printed in the cell above shows the information needed to build the matrices. However, Brightway also allows us to manipulate the `exchange` object and access its metadata.

In [ ]:
# Select the second `exchange` from the lista
exchange = list(seleccionado.technosphere())[1]
print(exchange.as_dict())

## Search options
As you can imagine, manipulating a database with so many activities (~21k) is quite complicated. We can use native Python functions to perform a search.

In [ ]:
for x in ei:
  if x['name'] == 'transport, freight, lorry >32 metric ton, EURO5':
    print(x)

In [ ]:
# truck = [x for x in ei if x['name'] == 'transport, freight, lorry >32 metric ton, EURO5'][0]
# truck

This way of searching is more *Pythonic*. However, you can also use Brightway's search engine through the `search` function.

In [ ]:
ei.search('transport, freight RoW >32 EURO5')

In [ ]:
ei.search?? # La funcion search prioriza algunos campos para hacer el filtro.

# Introduction to Brightway — Part 3

In this section, we will discuss the fundamental concepts of Brightway. It is important to note that all this information is available online in the documentation page:

https://docs.brightway.dev/en/latest/index.html

## Exploring the matrices
Now that we know how to create an activity and impact methods from scratch, we can focus on manipulating the activities already present in ecoinvent.
For this part, we will use a project prepared for you that contains a biosphere and technosphere compatible with ecoinvent v3.9.

In [ ]:
# Re-import for this section if running independently
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from rich import print

We have two databases.

In [ ]:
bd.databases

In [ ]:
# select the database ecoinvent y una actividad que tomaremos de ejemplo
ei = bd.Database("ecoinvent-3.9.1-cutoff")
harina = ei.search('fishmeal PE 65-67')[0]
harina

In [ ]:
# Choose a method que ya esta instalado y hace un LCA pero we stop at the LCI stage
method=('IPCC 2021', 'climate change', 'global warming potential (GWP100)')
lca = bc.LCA({harina:1},method=method) # Instancia la clase
lca.lci() # calcula el inventario de ciclo de vida

Remember that ecoinvent has 21,238 activities.
So, what dimensions should the technosphere matrix have?

In [ ]:
lca.technosphere_matrix.toarray().shape

What dimensions should the supply vector `s` have?

In [ ]:
lca.supply_array

If I wanted to know how much of 'anchovy caught in wooden vessels'
is required in TOTAL to produce 1 kg of fishmeal...

In [ ]:
anchoveta = ei.search('anchovy PE wooden')[1]
anchoveta

In [ ]:
# lca.activity_dict allows me to locate una actividad en la matriz.
lca.supply_array[lca.activity_dict[anchoveta.id]]

Now we continue with the LCIA.

In [ ]:
lca.lcia() # Calcula los impactos
print("The impact is: ", lca.score)

## Contribution analysis
To understand the different contributions, we need to keep using the LCA object.
This object keeps the LCA results in memory.

### Most important processes
To list the processes that generate the largest impacts, we will use the `bw2analyzer` and `pandas` packages.

In [ ]:
import pandas as pd
import bw2analyzer as ba
# ba.ContributionAnalysis().annotated_top_processes(lca=lca) # dificil de visulizar
ba.ContributionAnalysis.annotated_top_processes??

In [ ]:
pd.DataFrame(
    [(x, y, z["name"]) for x, y, z in ba.ContributionAnalysis().annotated_top_processes(lca=lca)],
    columns=["score", "quantity", "name"]
)

### Most important emissions
Similarly, we can obtain a ranking of the environmental flows that generate the largest impacts.

In [ ]:
import pandas as pd
import bw2analyzer as ba
pd.DataFrame(
    [(x, y, z["name"]) for x, y, z in ba.ContributionAnalysis().annotated_top_emissions(lca=lca)],
    columns=["score", "quantity", "name"]
)

The importance of emissions in the impact is related to both their amount and their characterization factors.
We can list these factors to inspect them.

In [ ]:
for key, cf in bd.Method(method).load():
    # print(key, cf)
    print(bd.Database('biosphere3').get(key[1]))